# TP Final — Módulo 1 | DataNova Argentina S.A.
**Alumno:** Troncoso, Leandro

**Diplomatura en Python para Ciencia de Datos y Machine Learning-Módulo I**

---

# Requisitos para ejecutar el notebook

## Archivos necesarios
Colocar los siguientes archivos en la misma carpeta que este notebook:
- `Ventas.xlsx`
- `Clientes.xlsx`
- `Productos.xlsx`

## Base de datos
Las preguntas 13 a 15 requieren una base de datos MySQL:

Asegurarse de que la configuracion sea la predeterminada de XAMPP:
   - Usuario: `root`
   - Contrasena: *(sin contrasena)*
   - Puerto: `3306`

> La base de datos `ventas_db` y la tabla `ventas` se crean automaticamente al ejecutar el notebook.

### Librerias requeridas
```
pip install pandas numpy requests sqlalchemy pymysql openpyxl
```

---

## PASO 1 — Importación y normalización de datos

In [57]:
import pandas as pd
import numpy as np
import requests
import unicodedata
import re
from io import StringIO
from sqlalchemy import create_engine, text

In [58]:
df_ventas    = pd.read_excel('Ventas.xlsx')
df_clientes  = pd.read_excel('Clientes.xlsx')
df_productos = pd.read_excel('Productos.xlsx')

### Carga de datos externos con pd.read_html
Usamos `requests` para simular un navegador real y `StringIO` para pasar el HTML como texto.

In [59]:
headers = {'User-Agent': 'Mozilla/5.0'}

url_pob = 'https://en.wikipedia.org/wiki/List_of_countries_and_dependencies_by_population'
html_pob = requests.get(url_pob, headers=headers).text
df_pob = pd.read_html(StringIO(html_pob))[0]

url_pib = 'https://www.worldometers.info/gdp/gdp-by-country/'
html_pib = requests.get(url_pib, headers=headers).text
df_pib = pd.read_html(StringIO(html_pib))[0]

url_rpc = 'https://en.wikipedia.org/wiki/List_of_countries_by_average_wage'
html_rpc = requests.get(url_rpc, headers=headers).text
df_rpc = pd.read_html(StringIO(html_rpc))[2]

print('Fuentes web cargadas.')

Fuentes web cargadas.


In [60]:
def normalizar_columna(col):
    col = str(col)
    col = col.lower()
    col = unicodedata.normalize('NFD', col)
    col = col.encode('ascii', 'ignore').decode('utf-8')
    col = re.sub(r'[^a-z0-9]+', '_', col)
    col = col.strip('_')
    return col

for df in [df_ventas, df_clientes, df_productos, df_pob, df_pib, df_rpc]:
    df.columns = [normalizar_columna(c) for c in df.columns]

print('Columnas normalizadas a snake_case.')

Columnas normalizadas a snake_case.


**PREGUNTA 1:** ¿Cuáles son las dimensiones (filas × columnas) de cada uno de los 6 dataframes?

In [61]:
nombres    = ['df_ventas', 'df_clientes', 'df_productos', 'df_pob', 'df_pib', 'df_rpc']
dataframes = [df_ventas,   df_clientes,   df_productos,   df_pob,   df_pib,   df_rpc]

for nombre, df in zip(nombres, dataframes):
    filas, cols = df.shape
    print(f"{nombre}: {filas} filas x {cols} columnas")

df_ventas: 9994 filas x 8 columnas
df_clientes: 793 filas x 8 columnas
df_productos: 1864 filas x 5 columnas
df_pob: 240 filas x 6 columnas
df_pib: 218 filas x 6 columnas
df_rpc: 13 filas x 2 columnas


**PREGUNTA 2:** ¿Existen valores NaN en los dataframes de ventas, clientes y productos?

In [62]:
print('=== NaN en df_ventas ===')
print(df_ventas.isnull().sum())

print('\n=== NaN en df_clientes ===')
print(df_clientes.isnull().sum())

print('\n=== NaN en df_productos ===')
print(df_productos.isnull().sum())

=== NaN en df_ventas ===
id_pedido          0
fecha_compra       3
fecha_envio        6
modo_envio         0
id_cliente         0
id_producto        0
precio_unitario    0
cantidad           0
dtype: int64

=== NaN en df_clientes ===
id_cliente          0
nombre_cliente      0
segmento            0
pais                0
ciudad            455
estado            455
codigo_postal     455
region            455
dtype: int64

=== NaN en df_productos ===
id_producto         0
categoria           0
sub_categoria       0
nombre_producto     1
coste_produccion    2
dtype: int64


---
## PASO 2 — Limpieza de inconsistencias
### 2.1 Borrar registros sin fecha de compra

**PREGUNTA 3:** ¿Cuántos registros se eliminaron de df_ventas por tener Fecha Compra vacía?

In [63]:
filas_antes = len(df_ventas)
df_ventas = df_ventas.dropna(subset=['fecha_compra']).reset_index(drop=True)
eliminados = filas_antes - len(df_ventas)

print(f"Registros eliminados por fecha_compra vacía: {eliminados}")

Registros eliminados por fecha_compra vacía: 3


### 2.2 Borrar registros duplicados

**PREGUNTA 4:** ¿Existían registros duplicados en df_ventas o df_productos? ¿Cuántos se eliminaron?

In [64]:
dup_ventas    = df_ventas.duplicated().sum()
dup_productos = df_productos.duplicated().sum()

print(f"Duplicados en df_ventas:    {dup_ventas}")
print(f"Duplicados en df_productos: {dup_productos}")

df_ventas    = df_ventas.drop_duplicates().reset_index(drop=True)
df_productos = df_productos.drop_duplicates().reset_index(drop=True)

Duplicados en df_ventas:    1
Duplicados en df_productos: 1


### 2.3 Inconsistencia en datos categóricos

**PREGUNTA 5:** ¿Qué valor incorrecto encontraste en la columna `segmento` de df_clientes? Corregilo usando `map()`.

In [65]:
print('Valores actuales en la columna segmento:')
print(df_clientes['segmento'].value_counts())

Valores actuales en la columna segmento:
segmento
Consumer       409
Corporate      235
Home Office    148
Corporates       1
Name: count, dtype: int64


In [66]:
# map() con diccionario — reemplaza valores específicos
# Ajustar el valor incorrecto segun lo que se observe arriba
mapa_segmento = {
    'Consumer':    'Consumer',
    'Corporate':   'Corporate',
    'Home Office': 'Home Office',
    'Corporates': 'Corporate',
}

df_clientes['segmento'] = df_clientes['segmento'].map(mapa_segmento)

print('Después de la corrección:')
print(df_clientes['segmento'].value_counts())

Después de la corrección:
segmento
Consumer       409
Corporate      236
Home Office    148
Name: count, dtype: int64


### 2.4 Conversión de tipos numéricos

**PREGUNTA 6:** La columna `precio_unitario` llegó como string con coma decimal. Convertila a float.

In [67]:
print(f"Tipo antes:  {df_ventas['precio_unitario'].dtype}")

# 1. Reemplazar coma decimal por punto  2. Convertir a float
df_ventas['precio_unitario'] = (
    df_ventas['precio_unitario']
    .astype(str)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

print(f"Tipo después: {df_ventas['precio_unitario'].dtype}")
print(df_ventas['precio_unitario'].head())

Tipo antes:  str
Tipo después: float64
0    261.9600
1    731.9400
2     14.6200
3    957.5775
4     22.3680
Name: precio_unitario, dtype: float64


### 2.5 Conversión de fechas

**PREGUNTA 7:** Las columnas `fecha_compra` y `fecha_envio` llegaron como object. Convertilas a datetime.

In [68]:
print(f"Tipos antes — fecha_compra: {df_ventas['fecha_compra'].dtype} | fecha_envio: {df_ventas['fecha_envio'].dtype}")

df_ventas['fecha_compra'] = pd.to_datetime(df_ventas['fecha_compra'])
df_ventas['fecha_envio']  = pd.to_datetime(df_ventas['fecha_envio'])

print(f"Tipos después — fecha_compra: {df_ventas['fecha_compra'].dtype} | fecha_envio: {df_ventas['fecha_envio'].dtype}")

Tipos antes — fecha_compra: datetime64[us] | fecha_envio: datetime64[us]
Tipos después — fecha_compra: datetime64[us] | fecha_envio: datetime64[us]


---
## PASO 3 — Limpieza estadística y outliers

**PREGUNTA 8:** Aplicá `describe()` sobre df_ventas. ¿Cuál es la mediana del `precio_unitario`? ¿Y el percentil 75 de `cantidad`?

In [69]:
display(df_ventas.describe())

mediana_precio = df_ventas['precio_unitario'].median()
p75_cantidad   = df_ventas['cantidad'].quantile(0.75)

print(f"Mediana de precio_unitario: {mediana_precio}")
print(f"Percentil 75 de cantidad:   {p75_cantidad}")

,fecha_compra,fecha_envio,precio_unitario,cantidad
count,9990,9984,9990.000000,9990.000000
mean,2020-04-30 04:00:00,2020-05-04 02:33:20.841346,229.834970,3.843544
min,2018-01-03 00:00:00,2018-01-07 00:00:00,0.000000,1.000000
25%,2019-05-23 00:00:00,2019-05-27 00:00:00,17.248000,2.000000
50%,2020-06-26 00:00:00,2020-06-29 00:00:00,54.384000,3.000000
75%,2021-05-14 00:00:00,2021-05-18 00:00:00,209.937500,5.000000
max,2021-12-30 00:00:00,2022-01-05 00:00:00,22638.480000,40.000000
std,NaN,NaN,623.352233,2.660570


Mediana de precio_unitario: 54.384
Percentil 75 de cantidad:   5.0


**PREGUNTA 9:** ¿Cuántos pedidos tienen más de 13 unidades? Eliminá esos registros.

In [70]:
pedidos_over_13 = (df_ventas['cantidad'] > 13).sum()
print(f"Pedidos con más de 13 unidades: {pedidos_over_13}")

df_ventas = df_ventas[df_ventas['cantidad'] <= 13].reset_index(drop=True)
print(f"Nuevas dimensiones de df_ventas: {df_ventas.shape}")

Pedidos con más de 13 unidades: 29
Nuevas dimensiones de df_ventas: (9961, 8)


**PREGUNTA 10:** Imputá los registros sin `fecha_envio` con `fecha_compra + mediana del plazo de entrega`. ¿Cuántos registros se imputaron?

> La diferencia entre dos columnas datetime da un timedelta. La `median()` de esos timedeltas se puede sumar directamente a una fecha.

In [71]:
df_ventas['plazo_entrega'] = df_ventas['fecha_envio'] - df_ventas['fecha_compra']
mediana_plazo = df_ventas['plazo_entrega'].dropna().median()
print(f"Mediana del plazo de entrega: {mediana_plazo}")

nan_antes = df_ventas['fecha_envio'].isnull().sum()
df_ventas['fecha_envio'] = df_ventas['fecha_envio'].fillna(
    df_ventas['fecha_compra'] + mediana_plazo
)
imputados = nan_antes - df_ventas['fecha_envio'].isnull().sum()

print(f"Registros imputados: {imputados}")

Mediana del plazo de entrega: 4 days 00:00:00
Registros imputados: 6


**PREGUNTA 11:** Usá `pd.cut()` para clasificar los pedidos: `'Económico'` (0-100), `'Estándar'` (100-500), `'Premium'` (más de 500).

In [72]:
df_ventas['rango_precio'] = pd.cut(
    df_ventas['precio_unitario'],
    bins=[0, 100, 500, float('inf')],
    labels=['Económico', 'Estándar', 'Premium']
)

print(df_ventas['rango_precio'].value_counts())

rango_precio
Económico    6217
Estándar     2589
Premium      1153
Name: count, dtype: int64


**PREGUNTA 12:** Usá `pd.qcut()` para dividir en 4 cuartiles. ¿Los cuartiles coinciden con los rangos del ejercicio anterior? ¿Por qué?

In [73]:
df_ventas['cuartil'] = pd.qcut(
    df_ventas['precio_unitario'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print(df_ventas['cuartil'].value_counts().sort_index())

_, limites = pd.qcut(df_ventas['precio_unitario'], q=4, retbins=True)
print('\nLímites de los cuartiles:')
for i in range(len(limites) - 1):
    print(f'  Q{i+1}: {limites[i]:.2f} — {limites[i+1]:.2f}')

cuartil
Q1    2491
Q2    2490
Q3    2490
Q4    2490
Name: count, dtype: int64

Límites de los cuartiles:
  Q1: 0.00 — 17.18
  Q2: 17.18 — 53.98
  Q3: 53.98 — 209.57
  Q4: 209.57 — 22638.48


**Respuesta:** No coinciden. `pd.cut()` usa rangos de valor **fijos** (0-100, 100-500, >500) definidos manualmente. `pd.qcut()` calcula los límites automáticamente para que **cada grupo tenga la misma cantidad de observaciones**. Si los datos no se distribuyen uniformemente en esos rangos, los límites serán distintos.

---
## PASO 3b — Carga a base de datos y lectura con SQL

> **Requisito:** Tener XAMPP corriendo con el servidor MySQL activo.

In [74]:
try:
    engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306')
    with engine.connect() as conn:
        conn.execute(text('CREATE DATABASE IF NOT EXISTS ventas_db'))
        conn.commit()
    engine = create_engine('mysql+pymysql://root:@127.0.0.1:3306/ventas_db')
    print('Conectado correctamente a ventas_db.')
except Exception as e:
    print(f"Error de conexión: {e}")
    print("Asegurate de que XAMPP esté corriendo.")

Conectado correctamente a ventas_db.


**PREGUNTA 13:** Cargá df_ventas limpio a la tabla `'ventas'` usando `to_sql()`. ¿Cuántos registros se cargaron?

In [75]:
df_para_cargar = df_ventas.drop(columns=['rango_precio', 'cuartil', 'plazo_entrega'], errors='ignore')

df_para_cargar.to_sql('ventas', engine, if_exists='replace', index=False)

print(f"Registros cargados a la tabla 'ventas': {len(df_para_cargar)}")

Registros cargados a la tabla 'ventas': 9961


**PREGUNTA 14:** Leé los datos de vuelta con `read_sql()` filtrando solo las ventas del año 2020 en adelante. ¿Cuántas filas trae la query vs el total cargado?

> Esto es **SQL pushdown**: el servidor filtra antes de enviar los datos.

In [76]:
df_ventas_sql = pd.read_sql("""
    SELECT *
    FROM ventas
    WHERE YEAR(fecha_compra) >= 2020
""", engine)

print(f"Filas traídas por la query (>= 2020): {len(df_ventas_sql)}")
print(f"Total cargado en la BD:                {len(df_para_cargar)}")

Filas traídas por la query (>= 2020): 5883
Total cargado en la BD:                9961


**PREGUNTA 15:** ¿Cuántos pedidos se registraron por año? Usá `GROUP BY` en la query.

In [77]:
df_por_anio = pd.read_sql("""
    SELECT
        YEAR(fecha_compra) AS anio,
        COUNT(*)           AS pedidos
    FROM ventas
    GROUP BY anio
    ORDER BY anio
""", engine)

print(df_por_anio)

   anio  pedidos
0  2018     1985
1  2019     2093
2  2020     2579
3  2021     3304


---
## PASO 4 — Consolidación y análisis

**PREGUNTA 16:** Combiná `df_ventas_sql`, `df_clientes` y `df_productos` en `df_global` con INNER JOIN. ¿Cuántas filas tiene el DataFrame resultante?

In [78]:
df_global = (
    df_ventas_sql
    .merge(df_clientes,  on='id_cliente',  how='inner')
    .merge(df_productos, on='id_producto', how='inner')
)

print(f"Filas en df_global: {len(df_global)}")
print(f"Columnas: {df_global.columns.tolist()}")

Filas en df_global: 5883
Columnas: ['id_pedido', 'fecha_compra', 'fecha_envio', 'modo_envio', 'id_cliente', 'id_producto', 'precio_unitario', 'cantidad', 'nombre_cliente', 'segmento', 'pais', 'ciudad', 'estado', 'codigo_postal', 'region', 'categoria', 'sub_categoria', 'nombre_producto', 'coste_produccion']


**PREGUNTA 17:** Creá las columnas `'ingreso'` y `'beneficio'`. ¿Cuál es la categoría con mayor beneficio total?

In [79]:
df_global['ingreso']   = df_global['precio_unitario'] * df_global['cantidad']
df_global['beneficio'] = df_global['ingreso'] - df_global['coste_produccion'] * df_global['cantidad']

beneficio_por_categoria = (
    df_global
    .groupby('categoria')['beneficio']
    .sum()
    .sort_values(ascending=False)
)

print(beneficio_por_categoria)
print(f"\nCategoría con mayor beneficio total: {beneficio_por_categoria.idxmax()}")

categoria
Technology         842616.632969
Office Supplies    819678.075197
Furniture          808753.872493
Name: beneficio, dtype: float64

Categoría con mayor beneficio total: Technology


**PREGUNTA 18:** Usá `transform()` para agregar `'beneficio_medio_categoria'`. ¿Cuántos pedidos superan la media de su categoría?

In [80]:
# transform() agrega la estadística del grupo a cada fila sin colapsar el DataFrame
df_global['beneficio_medio_categoria'] = (
    df_global.groupby('categoria')['beneficio'].transform('mean')
)

pedidos_sobre_media = (df_global['beneficio'] > df_global['beneficio_medio_categoria']).sum()

print(f"Pedidos que superan la media de su categoría: {pedidos_sobre_media}")
df_global[['categoria', 'beneficio', 'beneficio_medio_categoria']].head(8)

Pedidos que superan la media de su categoría: 1031


,categoria,beneficio,beneficio_medio_categoria
0,Furniture,63.723687,650.646720
1,Furniture,506.511811,650.646720
2,Office Supplies,5.747051,230.570485
3,Office Supplies,19.960169,230.570485
4,Office Supplies,53.019825,230.570485
5,Office Supplies,51.179380,230.570485
6,Office Supplies,NaN,230.570485
7,Furniture,6.597091,650.646720


**PREGUNTA 19:** Creá una `pivot_table` con el beneficio total por país (filas) y categoría (columnas). Incluí totales con `margins=True`.

In [81]:
tabla_pivot = pd.pivot_table(
    df_global,
    values='beneficio',
    index='pais',
    columns='categoria',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total'
)

tabla_pivot

categoria,Furniture,Office Supplies,Technology,Total
pais,,,,
Germany,175884.145302,357551.338062,277276.184586,8.107117e+05
Romania,12260.505668,29049.165474,15838.711338,5.714838e+04
Spain,183041.179095,79425.203569,157766.703277,4.202331e+05
United States,437568.042428,353652.368091,391735.033767,1.182955e+06
Total,808753.872493,819678.075197,842616.632969,2.471049e+06


**PREGUNTA 20:** Calculá la diferencia promedio en días entre `fecha_envio` y `fecha_compra` por país. ¿En qué país los tiempos son más largos?

In [82]:
df_global['dias_entrega'] = (df_global['fecha_envio'] - df_global['fecha_compra']).dt.days

dias_por_pais = (
    df_global
    .groupby('pais')['dias_entrega']
    .mean()
    .round(1)
    .sort_values(ascending=False)
)

print(dias_por_pais)
print(f"\nPaís con tiempos de entrega más largos: {dias_por_pais.idxmax()}")

pais
Germany          4.1
Romania          3.9
United States    3.9
Spain            3.7
Name: dias_entrega, dtype: float64

País con tiempos de entrega más largos: Germany


---
## PASO 5 — Reporte

**PREGUNTA 21:** Exportá `'Reporte_DataNova.xlsx'` con dos pestañas.

In [83]:
top_productos = (
    df_global
    .groupby('nombre_producto')['beneficio']
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

with pd.ExcelWriter('Reporte_DataNova.xlsx', engine='openpyxl') as writer:
    tabla_pivot.to_excel(writer, sheet_name='Beneficio_por_pais_cat')
    top_productos.to_excel(writer, sheet_name='Top_productos', index=False)

print("Archivo 'Reporte_DataNova.xlsx' exportado correctamente.")
print(f"  Pestaña 'Beneficio_por_pais_cat': {tabla_pivot.shape}")
print(f"  Pestaña 'Top_productos': {len(top_productos)} productos")

Archivo 'Reporte_DataNova.xlsx' exportado correctamente.
  Pestaña 'Beneficio_por_pais_cat': (5, 4)
  Pestaña 'Top_productos': 10 productos


In [84]:
engine.dispose()
print('Conexión cerrada correctamente.')

Conexión cerrada correctamente.
